# Pipeline completo de testes (sequencial)

Treino rigoroso + validação + pós-processamento + sweep fino automático no mesmo notebook.

In [ ]:
# 1) Dependências
!pip -q install tensorflow matplotlib pandas scipy

In [ ]:
# 2) Montar Drive (obrigatório)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Gravar script de treino
%%writefile treinamento_seguro_unet.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base

    raise RuntimeError(
        "Drive não montado. No Colab, execute antes: \n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


pasta_base = garantir_drive_montado()

KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 80
SEED = 42
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'


def localizar_tfrecord(base_dir: str) -> str:
    busca = glob.glob(os.path.join(base_dir, '*MASSIVE*tfrecord*'))
    if not busca:
        busca = glob.glob(os.path.join(base_dir, '*tfrecord*'))
        busca.sort(key=os.path.getmtime, reverse=True)
    if not busca:
        raise FileNotFoundError(f'Nenhum TFRecord encontrado em {base_dir}')
    return busca[0]


def normalizar_banda(img: tf.Tensor) -> tf.Tensor:
    min_v = tf.reduce_min(img)
    max_v = tf.reduce_max(img)
    return tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))


def parse_and_process(example_proto):
    features_dict = {
        band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
    }
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        img = normalizar_banda(img)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image_stacked, lbl


def augment(image, label):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        label = tf.image.flip_left_right(label)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        label = tf.image.flip_up_down(label)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    label = tf.image.rot90(label, k)
    return image, label


def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coef(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss


def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    u5 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = conv_block(u5, 128)

    u6 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = conv_block(u6, 64)

    u7 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = conv_block(u7, 32)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])


caminho_arquivo = localizar_tfrecord(pasta_base)
print(f"📂 Lendo dados de: {caminho_arquivo}")

print("🔢 Verificando tamanho do arquivo...")
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f"✅ Total de amostras: {N_REAL}")

N_TRAIN = int(N_REAL * 0.8)
full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(
    parse_and_process, num_parallel_calls=tf.data.AUTOTUNE
)
full_dataset = full_dataset.shuffle(max(N_REAL, 1), seed=SEED, reshuffle_each_iteration=False)

train_ds = full_dataset.take(N_TRAIN).map(augment, num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(max(N_TRAIN, 1), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(INPUT_BANDS)))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=bce_dice_loss,
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou'),
        dice_coef,
    ],
)

checkpoint_last = os.path.join(pasta_base, 'Modelo_Checkpoint_last.keras')
checkpoint_best = os.path.join(pasta_base, 'Modelo_Checkpoint_best.keras')
csv_log = os.path.join(pasta_base, 'historico_treinamento.csv')
final_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')

cbs = [
    callbacks.ModelCheckpoint(checkpoint_last, save_best_only=False, verbose=1),
    callbacks.ModelCheckpoint(checkpoint_best, monitor='val_iou', mode='max', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_iou', mode='max', patience=12, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.CSVLogger(csv_log),
]

print("🔥 Iniciando Retreinamento Rigoroso (foco em talhões)...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cbs,
)

model.save(final_path)
print(f"✅ SUCESSO! Modelo final salvo em: {final_path}")
print(f"✅ Melhor checkpoint salvo em: {checkpoint_best}")


In [ ]:
# 4) Executar treino
!python treinamento_seguro_unet.py

In [ ]:
# 5) Visualizar evolução do treino (loss / val_iou / dice)
import os
import pandas as pd
import matplotlib.pyplot as plt
csv_path = '/content/drive/MyDrive/Tese_IA_Jussara/historico_treinamento.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(1, 3, figsize=(18,4))
    if 'loss' in df.columns: ax[0].plot(df['loss'], label='train_loss')
    if 'val_loss' in df.columns: ax[0].plot(df['val_loss'], label='val_loss')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True)
    if 'iou' in df.columns: ax[1].plot(df['iou'], label='train_iou')
    if 'val_iou' in df.columns: ax[1].plot(df['val_iou'], label='val_iou')
    ax[1].set_title('IoU'); ax[1].legend(); ax[1].grid(True)
    if 'dice_coef' in df.columns: ax[2].plot(df['dice_coef'], label='train_dice')
    if 'val_dice_coef' in df.columns: ax[2].plot(df['val_dice_coef'], label='val_dice')
    ax[2].set_title('Dice'); ax[2].legend(); ax[2].grid(True)
    plt.tight_layout(); plt.show()
else:
    print(f'⚠️ CSV não encontrado: {csv_path}')

In [ ]:
# 6) Gravar script de validação visual
%%writefile validacao_visual_modelo.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import tensorflow as tf
import matplotlib.pyplot as plt


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base

    raise RuntimeError(
        "Drive não montado. No Colab, execute antes: \n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


print('--- INICIANDO PROVA REAL ---')
pasta_base = garantir_drive_montado()
caminho_modelo = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')

if not os.path.exists(caminho_modelo):
    raise FileNotFoundError(f'Modelo não encontrado: {caminho_modelo}')

model = tf.keras.models.load_model(caminho_modelo, compile=False)
print(f'✅ Modelo carregado: {caminho_modelo}')

busca_dados = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca_dados:
    raise FileNotFoundError('Nenhum TFRecord MASSIVE encontrado para validação visual.')

caminho_dados = busca_dados[0]
KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'


def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        min_v = tf.reduce_min(img)
        max_v = tf.reduce_max(img)
        img = tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image_stacked, lbl


dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(10).take(1)
imgs, labels = next(iter(dataset))
preds = model.predict(imgs, verbose=0)
THRESHOLD_INFERENCIA = 0.30
print(f'🎯 Limiar de inferência aplicado: {THRESHOLD_INFERENCIA:.2f}')
preds_bin = (preds > THRESHOLD_INFERENCIA).astype('float32')

intersection = (preds_bin * labels.numpy()).sum(axis=(1, 2, 3))
union = ((preds_bin + labels.numpy()) > 0).sum(axis=(1, 2, 3))
iou = (intersection + 1e-6) / (union + 1e-6)
print(f'📏 IoU médio no lote: {iou.mean():.4f}')

plt.figure(figsize=(18, 14))
print('\nLEGENDA: Esquerda=Satélite | Meio=Gabarito | Direita=Predição binária')

n_show = min(5, imgs.shape[0])
for i in range(n_show):
    plt.subplot(n_show, 3, i * 3 + 1)
    plt.imshow(imgs[i][:, :, 2], cmap='RdYlGn', vmin=0, vmax=1)
    plt.axis('off')
    if i == 0:
        plt.title('Satélite (NDVI)')

    plt.subplot(n_show, 3, i * 3 + 2)
    plt.imshow(labels[i][:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0:
        plt.title('Gabarito Real')

    plt.subplot(n_show, 3, i * 3 + 3)
    plt.imshow(preds_bin[i][:, :, 0], cmap='viridis')
    plt.axis('off')
    if i == 0:
        plt.title(f'Predição (>{THRESHOLD_INFERENCIA:.2f})')

saida_fig = os.path.join(pasta_base, 'prova_real_validacao.png')
plt.tight_layout()
plt.savefig(saida_fig, dpi=200, bbox_inches='tight')
plt.show()
print(f'🖼️ Prova real salva em: {saida_fig}')


In [ ]:
# 7) Executar prova real
!python validacao_visual_modelo.py

In [ ]:
# 8) Exibir imagem da prova real
from IPython.display import Image, display
import os
img_path = '/content/drive/MyDrive/Tese_IA_Jussara/prova_real_validacao.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
    print(f'✅ Exibindo: {img_path}')
else:
    print(f'⚠️ Imagem não encontrada em: {img_path}')

In [ ]:
# 9) Gravar script de pós-processamento
%%writefile pos_processamento_mascara.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import ndimage as ndi


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base
    raise RuntimeError(
        "Drive não montado. No Colab, execute antes:\n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


def limpar_mascara(mask_bin: np.ndarray, min_size: int = 100) -> np.ndarray:
    mask = mask_bin.astype(bool)
    mask = ndi.binary_opening(mask, structure=np.ones((3, 3), dtype=bool))
    mask = ndi.binary_closing(mask, structure=np.ones((3, 3), dtype=bool))

    labeled, n = ndi.label(mask)
    if n == 0:
        return mask.astype(np.uint8)

    counts = np.bincount(labeled.ravel())
    remove = counts < min_size
    remove[0] = False
    mask[remove[labeled]] = False
    return mask.astype(np.uint8)


def metricas(y_true: np.ndarray, y_pred: np.ndarray):
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    return precision, recall, f1, iou


pasta_base = garantir_drive_montado()
modelo_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')
if not os.path.exists(modelo_path):
    raise FileNotFoundError(f'Modelo não encontrado: {modelo_path}')

busca = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca:
    busca = glob.glob(os.path.join(pasta_base, '*tfrecord*'))
if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord encontrado em: {pasta_base}')

caminho_dados = sorted(busca, key=os.path.getmtime, reverse=True)[0]
print(f'📂 Dados usados: {caminho_dados}')

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
THRESHOLD = float(os.environ.get('THRESHOLD_INFERENCIA', '0.30'))
MIN_SIZE = int(os.environ.get('MIN_COMPONENT_SIZE', '100'))


def parse_fast(example_proto):
    features = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features)
    xs = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        min_v = tf.reduce_min(img)
        max_v = tf.reduce_max(img)
        img = tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))
        xs.append(img)
    image = tf.concat(xs, axis=-1)

    lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image, lbl


model = tf.keras.models.load_model(modelo_path, compile=False)
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(10).take(1)
imgs, labels = next(iter(dataset))
probs = model.predict(imgs, verbose=0)
raw = (probs > THRESHOLD).astype(np.uint8)

clean = np.zeros_like(raw)
for i in range(raw.shape[0]):
    clean[i, :, :, 0] = limpar_mascara(raw[i, :, :, 0], min_size=MIN_SIZE)

y_true = labels.numpy().astype(np.uint8)

p_raw, r_raw, f_raw, i_raw = metricas(y_true, raw)
p_cln, r_cln, f_cln, i_cln = metricas(y_true, clean)

print('📊 Métricas antes do pós-processamento:')
print(f'precision={p_raw:.4f} recall={r_raw:.4f} f1={f_raw:.4f} iou={i_raw:.4f}')
print('📊 Métricas depois do pós-processamento:')
print(f'precision={p_cln:.4f} recall={r_cln:.4f} f1={f_cln:.4f} iou={i_cln:.4f}')

plt.figure(figsize=(18, 16))
n_show = min(4, imgs.shape[0])
for i in range(n_show):
    plt.subplot(n_show, 4, i*4 + 1)
    plt.imshow(imgs[i][:,:,2], cmap='RdYlGn', vmin=0, vmax=1)
    plt.axis('off')
    if i == 0: plt.title('Satélite NDVI')

    plt.subplot(n_show, 4, i*4 + 2)
    plt.imshow(y_true[i,:,:,0], cmap='binary_r')
    plt.axis('off')
    if i == 0: plt.title('Gabarito')

    plt.subplot(n_show, 4, i*4 + 3)
    plt.imshow(raw[i,:,:,0], cmap='magma')
    plt.axis('off')
    if i == 0: plt.title(f'Bruta > {THRESHOLD:.2f}')

    plt.subplot(n_show, 4, i*4 + 4)
    plt.imshow(clean[i,:,:,0], cmap='viridis')
    plt.axis('off')
    if i == 0: plt.title(f'Pós-processada (min_size={MIN_SIZE})')

plt.tight_layout()
out_img = os.path.join(pasta_base, 'comparativo_pos_processamento.png')
plt.savefig(out_img, dpi=200, bbox_inches='tight')
plt.show()
print(f'🖼️ Comparativo salvo em: {out_img}')

out_metrics = os.path.join(pasta_base, 'resultado_pos_processamento.txt')
with open(out_metrics, 'w', encoding='utf-8') as f:
    f.write('stage\tprecision\trecall\tf1\tiou\n')
    f.write(f'antes\t{p_raw:.6f}\t{r_raw:.6f}\t{f_raw:.6f}\t{i_raw:.6f}\n')
    f.write(f'depois\t{p_cln:.6f}\t{r_cln:.6f}\t{f_cln:.6f}\t{i_cln:.6f}\n')
    f.write(f'threshold={THRESHOLD:.2f}\n')
    f.write(f'min_component_size={MIN_SIZE}\n')
print(f'📝 Métricas do pós-processamento salvas em: {out_metrics}')


In [ ]:
# 10) Rodar pós-processamento (limpeza de máscara)
!python pos_processamento_mascara.py

In [ ]:
# 11) Exibir comparativo do pós-processamento
from IPython.display import Image, display
import os
img_path = '/content/drive/MyDrive/Tese_IA_Jussara/comparativo_pos_processamento.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
    print(f'✅ Exibindo: {img_path}')
else:
    print(f'⚠️ Imagem não encontrada em: {img_path}')

In [ ]:
# 12) Gravar script de sweep de limiares
%%writefile teste_limiares_talhoes.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import numpy as np
import tensorflow as tf


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base
    raise RuntimeError(
        "Drive não montado. No Colab, execute antes:\n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


def parse_thresholds(default):
    csv = os.environ.get('THRESHOLDS_CSV', '')
    if not csv.strip():
        return default
    vals = []
    for x in csv.split(','):
        x = x.strip()
        if x:
            vals.append(float(x))
    return vals if vals else default


pasta_base = garantir_drive_montado()
modelo_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')
if not os.path.exists(modelo_path):
    raise FileNotFoundError(f'Modelo não encontrado: {modelo_path}')

busca = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca:
    busca = glob.glob(os.path.join(pasta_base, '*tfrecord*'))
if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord encontrado em: {pasta_base}')

caminho_dados = sorted(busca, key=os.path.getmtime, reverse=True)[0]
print(f'📂 Dados usados no teste: {caminho_dados}')

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
BATCH_SIZE = int(os.environ.get('BATCH_SIZE_SWEEP', '16'))
N_BATCHES = int(os.environ.get('N_BATCHES_SWEEP', '20'))
THRESHOLDS = parse_thresholds([0.30, 0.40, 0.50, 0.60, 0.70])
OUTPUT_FILENAME = os.environ.get('SWEEP_OUTPUT_FILENAME', 'resultado_teste_limiares.txt')


def parse_fast(example_proto):
    features = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features)

    xs = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        min_v = tf.reduce_min(img)
        max_v = tf.reduce_max(img)
        img = tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))
        xs.append(img)

    image = tf.concat(xs, axis=-1)
    lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image, lbl


def metricas_binarias(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(np.uint8)
    y_true = y_true.astype(np.uint8)

    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    return precision, recall, f1, iou


print('🤖 Carregando modelo...')
model = tf.keras.models.load_model(modelo_path, compile=False)

print('📦 Preparando lote de validação para sweep de limiar...')
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(BATCH_SIZE).take(N_BATCHES)

all_probs, all_labels = [], []
for imgs, labels in dataset:
    probs = model.predict(imgs, verbose=0)
    all_probs.append(probs)
    all_labels.append(labels.numpy())

if not all_probs:
    raise RuntimeError('Nenhum batch lido para teste de limiares.')

y_prob = np.concatenate(all_probs, axis=0)
y_true = np.concatenate(all_labels, axis=0)

print(f'✅ Amostras avaliadas: {y_prob.shape[0]}')
print('\n📊 Resultado por limiar:')
print('thr\tprecision\trecall\tf1\tiou')

best = None
for thr in THRESHOLDS:
    p, r, f1, iou = metricas_binarias(y_true, y_prob, thr)
    print(f'{thr:.2f}\t{p:.4f}\t\t{r:.4f}\t{f1:.4f}\t{iou:.4f}')
    if best is None or f1 > best['f1']:
        best = {'thr': thr, 'precision': p, 'recall': r, 'f1': f1, 'iou': iou}

print('\n🏆 Melhor limiar (por F1):')
print(best)

out_path = os.path.join(pasta_base, OUTPUT_FILENAME)
with open(out_path, 'w', encoding='utf-8') as f:
    f.write('thr\tprecision\trecall\tf1\tiou\n')
    for thr in THRESHOLDS:
        p, r, f1, iou = metricas_binarias(y_true, y_prob, thr)
        f.write(f'{thr:.2f}\t{p:.6f}\t{r:.6f}\t{f1:.6f}\t{iou:.6f}\n')
    f.write(f"\nmelhor_thr={best['thr']:.2f}\n")

print(f'📝 Resultado salvo em: {out_path}')


In [ ]:
# 13) Sweep grosso
!python teste_limiares_talhoes.py

In [ ]:
# 14) Sweep fino automático ao redor do melhor threshold
import os
res_path = '/content/drive/MyDrive/Tese_IA_Jussara/resultado_teste_limiares.txt'
best = 0.30
if os.path.exists(res_path):
    with open(res_path, 'r', encoding='utf-8') as f:
        for ln in f:
            if ln.startswith('melhor_thr='):
                best = float(ln.split('=')[1].strip())
                break
vals = [max(0.05, min(0.95, best + d)) for d in [-0.10, -0.08, -0.06, -0.04, -0.02, 0.0, 0.02, 0.04, 0.06, 0.08, 0.10]]
vals = sorted(set(round(v,2) for v in vals))
csv = ','.join(f'{v:.2f}' for v in vals)
print('🔎 Limiar base:', best)
print('🔎 Sweep fino:', csv)
os.environ['THRESHOLDS_CSV'] = csv
os.environ['SWEEP_OUTPUT_FILENAME'] = 'resultado_teste_limiares_fino.txt'
!python teste_limiares_talhoes.py

In [ ]:
# 15) Bloco único final: métricas antes/depois + visual + recomendação automática
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

base = '/content/drive/MyDrive/Tese_IA_Jussara'
res_pos_path = os.path.join(base, 'resultado_pos_processamento.txt')
img_pos_path = os.path.join(base, 'comparativo_pos_processamento.png')
coarse_path = os.path.join(base, 'resultado_teste_limiares.txt')
fine_path = os.path.join(base, 'resultado_teste_limiares_fino.txt')


def load_sweep(path):
    if not os.path.exists(path):
        return None, None
    with open(path, 'r', encoding='utf-8') as f:
        txt = f.read()
    linhas = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    dados, best = [], None
    for ln in linhas[1:]:
        if ln.startswith('melhor_thr='):
            best = float(ln.split('=')[1])
            break
        p = ln.split('	')
        if len(p) == 5:
            dados.append(p)
    if not dados:
        return None, best
    df = pd.DataFrame(dados, columns=['thr', 'precision', 'recall', 'f1', 'iou'])
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df, best


def load_pos_metrics(path):
    if not os.path.exists(path):
        return None, None, None
    with open(path, 'r', encoding='utf-8') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    rows = []
    threshold = None
    min_component_size = None
    for ln in lines[1:]:
        if ln.startswith('threshold='):
            threshold = float(ln.split('=')[1])
            continue
        if ln.startswith('min_component_size='):
            min_component_size = int(ln.split('=')[1])
            continue
        p = ln.split('	')
        if len(p) == 5:
            rows.append(p)
    if not rows:
        return None, threshold, min_component_size
    df = pd.DataFrame(rows, columns=['stage', 'precision', 'recall', 'f1', 'iou'])
    for c in ['precision', 'recall', 'f1', 'iou']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df, threshold, min_component_size


print('=== 1) Métricas antes/depois do pós-processamento ===')
df_pos, thr_pos, min_size = load_pos_metrics(res_pos_path)
if df_pos is None:
    print(f'⚠️ Arquivo não encontrado ou inválido: {res_pos_path}')
else:
    display(df_pos)
    if {'antes', 'depois'}.issubset(set(df_pos['stage'])):
        b = df_pos[df_pos['stage'] == 'antes'].iloc[0]
        a = df_pos[df_pos['stage'] == 'depois'].iloc[0]
        print('Δ (depois - antes):')
        print(f"precision: {a['precision'] - b['precision']:+.4f}")
        print(f"recall:    {a['recall'] - b['recall']:+.4f}")
        print(f"f1:        {a['f1'] - b['f1']:+.4f}")
        print(f"iou:       {a['iou'] - b['iou']:+.4f}")

print('\n=== 2) Comparação visual lado a lado ===')
if os.path.exists(img_pos_path):
    display(Image(filename=img_pos_path))
    print(f'🖼️ Exibindo: {img_pos_path}')
else:
    print(f'⚠️ Imagem não encontrada: {img_pos_path}')

print('\n=== 3) Recomendação automática do melhor conjunto ===')
dfc, bc = load_sweep(coarse_path)
dff, bf = load_sweep(fine_path)
if dff is not None:
    best_thr = bf
    src = 'sweep_fino'
    sweep_df = dff
elif dfc is not None:
    best_thr = bc
    src = 'sweep_grosso'
    sweep_df = dfc
else:
    best_thr = thr_pos if thr_pos is not None else 0.30
    src = 'fallback_pos_processamento'
    sweep_df = None

usar_pos = False
if df_pos is not None and {'antes', 'depois'}.issubset(set(df_pos['stage'])):
    f1_antes = float(df_pos[df_pos['stage'] == 'antes']['f1'].iloc[0])
    f1_depois = float(df_pos[df_pos['stage'] == 'depois']['f1'].iloc[0])
    usar_pos = f1_depois >= f1_antes

recomendacao = {
    'threshold_inferencia': round(float(best_thr), 2),
    'usar_pos_processamento': bool(usar_pos),
    'min_component_size': int(min_size) if min_size is not None else 100,
    'origem_threshold': src,
}

print('✅ Conjunto recomendado:')
for k, v in recomendacao.items():
    print(f'- {k}: {v}')

if sweep_df is not None:
    display(sweep_df)
    plt.figure(figsize=(8, 4))
    for met in ['precision', 'recall', 'f1', 'iou']:
        plt.plot(sweep_df['thr'], sweep_df[met], marker='o', label=met)
    plt.axvline(recomendacao['threshold_inferencia'], color='red', linestyle='--', label='thr recomendado')
    plt.xlabel('Threshold')
    plt.ylabel('Métrica')
    plt.title(f'Métricas por threshold ({src})')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

out_rec = os.path.join(base, 'recomendacao_pipeline.txt')
with open(out_rec, 'w', encoding='utf-8') as f:
    for k, v in recomendacao.items():
        f.write(f'{k}={v}\n')
print(f'📝 Recomendação salva em: {out_rec}')
